Correcting seqs before inserting into Smartsheet as I added amino acids to end of the seqs to meet Twist DNA minimum length

***Please refer to this for round1 collagen designs received on May 8th 2026***

In [0]:
import pandas as pd
import biotite 
import biotite.structure as struc
import biotite.sequence as seq

In [0]:
df_designs = pd.read_csv("/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_ordered_designs/april_24_dble_strand_collagen_ordered_designs.csv", index_col = 0)
df_designs

In [0]:
rank1_dna = seq.NucleotideSequence("ATGGAGCTGGTTGTGCATGGTGCCGCACCAGAGGAAGCCCCGCTCGTGGAGGAACTGAGAGAATTTGCGGAAGCAGAAGATAAACGGATCTACGAAGAACGCGAAGCCGAGATTGAAGAACTGTATGCAAGAATTACCGCGCTGCGCGAAGAGGGCTCGAAAAACGCTCCTCCGGGCTTCCCAGGTGAACGGGGGCCCGATGTAGAAGTGGGGACTGTTACACGTGCCGAGTTATTGGCCGATCCTGAGGGTACATTGGCCGAGTTAAAAGCAGCAATACGGGAAGCGATTCGAGCCCAGCATGAAGCGACAGTGGCGGAACTGCGTGCCCATGCAGCAGCAGCAGAAGCTGGGCTGAAAACAGGAGTGCCACCAGGCTTTCCAGGTGAGCGTGGCGATCTGGTTGTTAGAATTCTGCCG")
rank1_dna.translate(complete= True)

In [0]:
rank1_protein = df_designs['sequence'].iloc[0]
rank1_protein_twist = rank1_dna.translate(complete= True)
assert(rank1_protein == str(rank1_protein_twist))

In [0]:
import os
import biotite.sequence.io.fasta as fasta
import biotite.sequence as seq

path_folder_twist_dna = "/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_ordered_designs/round1_dbl_strand_collagen_twist_seqs/"
twist_headers = []
twist_seqs = []
twist_ranks = []
for filename in os.listdir(path_folder_twist_dna):
    if filename.endswith(".fasta"):
        path = os.path.join(path_folder_twist_dna, filename)
        fasta_file = fasta.FastaFile.read(path)
        for header, sequence in fasta_file.items():
            twist_headers.append(header)
            twist_rank = filename.split('_')[-1].split('.')[0]
            twist_ranks.append(twist_rank)
            twist_seqs.append(sequence)
    else:
        continue


df_twist_seqs = pd.DataFrame({'rank': twist_ranks, 'header': twist_headers, 'seq_dna': twist_seqs})
df_twist_seqs['seq_protein_twist'] = df_twist_seqs['seq_dna'].apply(lambda x: ''.join(seq.NucleotideSequence(x).translate(complete= True)))
df_twist_seqs

In [0]:
df_designs_seqs = df_designs[['design_name_simple', 'sequence', 'sequence_with_tags']]
df_designs_seqs['rank'] = df_designs_seqs['design_name_simple'].apply(lambda x: x.split('_')[-1])
df_designs_seqs

In [0]:
df_twist_designs = pd.merge(df_twist_seqs, df_designs_seqs, on = 'rank', how = 'left')
df_twist_designs

In [0]:
match_ids = []
for i in range(len(df_twist_designs)):
    if df_twist_designs['seq_protein_twist'].iloc[i] == df_twist_designs['sequence'].iloc[i]:
        match_ids.append(i)
print("Match IDs: ", match_ids)
mismatch_ids = list(set(range(len(df_twist_designs))) - set(match_ids))
print("Mismatch IDs: ", mismatch_ids)
print("-" * 150)
for index in mismatch_ids:
    print("Index: ", index)
    print("Rank: ", df_twist_designs['rank'].iloc[index])
    print("Design Name:  ",df_twist_designs['design_name_simple'].iloc[index])
    print("Designed Seq: ", df_twist_designs['sequence'].iloc[index])
    print("Twist Seq:    ", df_twist_designs['seq_protein_twist'].iloc[index])
    print("-" * 50)

In [0]:
tag_prefix = 'M'
tag_suffix = 'LE' + ("H" *6)
df_twist_designs_copy = df_twist_designs.copy()
df_twist_designs_copy['seq_protein_twist_no_stop_codon'] = df_twist_designs_copy['seq_protein_twist'].apply(lambda x: x.split('*')[0])
df_twist_designs_copy['seq_protein_twist_no_stop_codon_with_prefix'] = df_twist_designs_copy['seq_protein_twist_no_stop_codon'].apply(lambda x: tag_prefix + x)
df_twist_designs_copy['seq_protein_twist_no_stop_codon_with_prefix_and_suffix'] = df_twist_designs_copy['seq_protein_twist_no_stop_codon_with_prefix'].apply(lambda x: x + tag_suffix if tag_suffix not in x else x)
df_twist_designs_copy

In [0]:
for index in range(len(df_twist_designs_copy)):
    print("Index: ", index)
    print("Rank: ", df_twist_designs['rank'].iloc[index])
    print("Design Name:  ",df_twist_designs['design_name_simple'].iloc[index])
    print("Designed Seq: ", df_twist_designs['sequence'].iloc[index])
    print("Twist Seq:    ", df_twist_designs['seq_protein_twist'].iloc[index])
    print("Twist Seq Fn: ", df_twist_designs_copy['seq_protein_twist_no_stop_codon_with_prefix_and_suffix'].iloc[index])
    print("-" * 50)

In [0]:
df_twist_designs_copy['rank_id'] = df_twist_designs_copy['rank'].apply(lambda x: int(x.split('rank')[-1]))
df_twist_designs_copy.sort_values(by = 'rank_id', ascending = True, inplace = True)
df_twist_designs_final = df_twist_designs_copy.drop(columns = ['rank', 'sequence_with_tags', 'design_name_simple', 'seq_protein_twist_no_stop_codon', 'seq_protein_twist_no_stop_codon_with_prefix'])[['rank_id', 'header', 'seq_protein_twist_no_stop_codon_with_prefix_and_suffix', 'seq_dna', 'seq_protein_twist']]
df_twist_designs_final.rename(columns = {'seq_protein_twist_no_stop_codon_with_prefix_and_suffix': 'seq_protein_twist_final'}, inplace= True)
#df_twist_designs_final.to_csv("/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_ordered_designs/april_24_dble_strand_collagen_correct_twist_seqs.csv")